# 05 · Evaluation — the five questions

This notebook reads whatever `results/` contains and answers the questions from
the project brief.  Run the three stage scripts first:

```
uv run python scripts/run_stage1.py
uv run python scripts/run_stage2.py
uv run python scripts/run_stage3.py
```

In [ ]:
# Section 1: Setup
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "configs" / "default.yaml").exists():
    ROOT = ROOT.parent          # running from notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from configs.loader import load_config, build_victim, resolve

cfg = load_config()
print("project root:", ROOT)
print("victim config:", cfg["victim"])

In [ ]:
import json
import pandas as pd

RESULTS = resolve(cfg["output"]["results_dir"])
FIGURES = resolve(cfg["output"]["figures_dir"])

frames = {}
for stage in ["stage1", "stage2", "stage3"]:
    path = RESULTS / stage / "summary.json"
    if path.exists():
        df = pd.DataFrame(json.loads(path.read_text()))
        df["stage_name"] = stage
        frames[stage] = df
    else:
        print(f"missing: {path}")

summary = pd.concat(frames.values(), ignore_index=True) if frames else pd.DataFrame()
summary[["stage_name", "label", "n_episodes", "attack_success_rate",
         "mean_baseline_confidence", "mean_best_confidence", "mean_confidence_drop",
         "mean_reward", "mean_movement_cost", "mean_episode_length"]].round(3)

## Section 2–3: Environments and what they look like

In [ ]:
from IPython.display import Image as ShowImage, display

for name in ["stage1_best_attack.png", "stage2_no_obstacle_best_attack.png",
             "stage3_no_obstacle_best_attack.png"]:
    path = FIGURES / name
    if path.exists():
        print(name)
        display(ShowImage(filename=str(path), width=760))

## Section 4–5: Baselines vs PPO

**Q3 — is PPO better than Random / Greedy?**

In [ ]:
if not summary.empty:
    view = summary[summary.stage_name != "stage1"]
    pivot = view.pivot_table(index="method", columns="variant",
                             values=["attack_success_rate", "mean_confidence_drop", "mean_reward"])
    display(pivot.round(3))

**Q2 / Q4 — what do physics and obstacles cost the attacker?**

In [ ]:
if not summary.empty:
    display(summary.pivot_table(index=["stage_name", "variant"], columns="method",
                                values="attack_success_rate").round(3))
    display(summary.pivot_table(index=["stage_name", "variant"], columns="method",
                                values="mean_confidence_drop").round(3))

## Section 6: Evaluation — the five questions

In [ ]:
def answer(question, condition, yes, no):
    print(f"{question}\n  -> {yes if condition else no}\n")

if not summary.empty:
    s1 = summary[summary.stage_name == "stage1"]
    s2 = summary[summary.stage_name == "stage2"]
    s3 = summary[summary.stage_name == "stage3"]

    answer("Q1  Can the agent affect the victim through a restricted API?",
           len(s1) and s1.mean_confidence_drop.max() > 0.05,
           f"yes - stage 1 confidence drop up to {s1.mean_confidence_drop.max():.3f}",
           "no measurable effect")

    if len(s1) and len(s2):
        answer("Q2  What do physical constraints cost?",
               True,
               f"stage1 success {s1.attack_success_rate.max():.2f} vs "
               f"stage2 success {s2.attack_success_rate.max():.2f}",
               "")

    if len(s2):
        ppo = s2[s2.method == "ppo"].mean_reward.mean()
        base = s2[s2.method != "ppo"].mean_reward.mean()
        answer("Q3  Is PPO better than the baselines?",
               ppo > base,
               f"yes - mean reward {ppo:.3f} vs {base:.3f}",
               f"not in this budget - mean reward {ppo:.3f} vs {base:.3f}")

    if len(s2) and s2.variant.nunique() > 1:
        with_obs = s2[s2.variant == "obstacle"].attack_success_rate.mean()
        without = s2[s2.variant == "no_obstacle"].attack_success_rate.mean()
        answer("Q4  Does the agent adapt to obstacles?",
               with_obs > 0.0,
               f"success {without:.2f} without obstacles vs {with_obs:.2f} with",
               "obstacles defeated the attack entirely")

    answer("Q5  Does the same Agent/API design survive 2D -> 3D?",
           len(s3) > 0,
           "yes - stage 3 ran the same agents, reward and driver unchanged",
           "stage 3 has not been run yet")

## Section 7: Visualization

In [ ]:
for name in sorted(p.name for p in FIGURES.glob("*.png")):
    print(name)
    display(ShowImage(filename=str(FIGURES / name), width=900))